# SENTINEL — Multi-Agent RL for Cloud Incident Response
### OpenEnv Hackathon 2026 | Meta PyTorch

| Cell | What it does |
|------|-------------|
| 1 | Install dependencies |
| 2 | Instantiate environment |
| 3 | **Baseline** — random actions, 100 episodes |
| 4 | **Trained** — UCB1 + Bayesian RCA, 100 episodes |
| 5 | Plot 4-panel training curves |
| 6 | Before/After behavior transcript |
| 7 | Summary table |

**Math algorithms (no LLM, no API keys):**
- UCB1 bandit (Auer et al. 2002)
- Bayesian Noisy-OR (Pearl 1988 / MicroRank 2021)
- Personalized PageRank (Brin & Page 1998)
- ALP Curriculum (Portelas et al. CoRL 2020)

> **No GPU required.** All cells run on CPU.

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys, os
if not os.path.exists('sentinel'):
    subprocess.run(['git', 'clone', 'https://github.com/SayantikaLaskar/sentinel.git'], check=False)
    os.chdir('sentinel')
elif os.path.basename(os.getcwd()) != 'sentinel':
    os.chdir('sentinel')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Dependencies installed')

## Cell 2 — Instantiate Environment

In [ ]:
import sys, os
if '.' not in sys.path: sys.path.insert(0, '.')
from sentinel.env import Sentinel_Env
env = Sentinel_Env(config_path='env_spec.yaml', incident_library_path='incident_library.yaml', render_mode='human')
obs, info = env.reset(seed=42)
print(f'Incident: {info["incident_id"]}')
print(env.render())
print('Observation keys:', list(obs.keys()))
print('Observation space:', env.observation_space)
print('Action space:     ', env.action_space)

## Cell 3 — Baseline (Random Actions) — 100 Episodes

In [ ]:
import random
from sentinel.env import Sentinel_Env

SEED, N = 0, 100
random.seed(SEED)
baseline_env = Sentinel_Env(config_path='env_spec.yaml', incident_library_path='incident_library.yaml')

RANDOM_ACTIONS = [
    {'agent': 'holmes', 'category': 'investigative', 'name': 'QueryLogs',
     'params': {'service': 'api-gateway', 'time_range': [0, 60]}},
    {'agent': 'holmes', 'category': 'investigative', 'name': 'QueryMetrics',
     'params': {'service': 'web-gateway', 'metric_name': 'cpu', 'time_range': [0, 300]}},
    {'agent': 'forge', 'category': 'remediation', 'name': 'RestartService',
     'params': {'service': 'api-gateway'}},
    {'agent': 'forge', 'category': 'remediation', 'name': 'ScaleService',
     'params': {'service': 'web-gateway', 'replicas': 2}},
]

baseline_rewards = []
for ep in range(N):
    obs, info = baseline_env.reset(seed=SEED + ep)
    terminated = truncated = False
    ep_reward, step = 0.0, 0
    while not (terminated or truncated) and step < 60:
        obs, r, terminated, truncated, _ = baseline_env.step(RANDOM_ACTIONS[step % len(RANDOM_ACTIONS)])
        ep_reward += r; step += 1
    baseline_rewards.append(ep_reward)
    if (ep + 1) % 20 == 0:
        print(f'Episode {ep+1:3d} | avg(last 20) = {sum(baseline_rewards[-20:])/20:.4f}')

print(f'\nBaseline mean reward: {sum(baseline_rewards)/len(baseline_rewards):.4f}')

## Cell 4 — Trained Agent (UCB1 + Bayesian RCA) — 100 Episodes

- **UCB1 bandit** (Auer et al. 2002) selects action type
- **Bayesian Noisy-OR** (Pearl 1988) identifies root-cause service

No LLM, no API keys — pure math.

In [ ]:
from sentinel.env import Sentinel_Env
from sentinel.training.pipeline import _get_action, TrainingConfig
from sentinel.math_engine import get_ucb1_bandit

SEED, N = 100, 100
trained_env = Sentinel_Env(config_path='env_spec.yaml', incident_library_path='incident_library.yaml')
cfg = TrainingConfig(agent='holmes')
bandit = get_ucb1_bandit()

trained_rewards = []
for ep in range(N):
    obs, info = trained_env.reset(seed=SEED + ep)
    terminated = truncated = False
    ep_reward, step = 0.0, 0
    while not (terminated or truncated) and step < 60:
        action = _get_action(None, obs, cfg)
        arm_idx = action.pop('_ucb1_arm_idx', None)
        obs, r, terminated, truncated, _ = trained_env.step(action)
        if arm_idx is not None: bandit.update(arm_idx, float(r))
        ep_reward += r; step += 1
    trained_rewards.append(ep_reward)
    if (ep + 1) % 20 == 0:
        print(f'Episode {ep+1:3d} | avg(last 20) = {sum(trained_rewards[-20:])/20:.4f}')

print(f'\nTrained mean reward: {sum(trained_rewards)/len(trained_rewards):.4f}')
print('\nUCB1 Top Arms:')
for s in bandit.arm_stats()[:5]:
    print(f'  {s["arm"]}: pulls={s["pulls"]}, mean_reward={s["mean_reward"]}')

## Cell 5 — Plot Training Curves (4-panel)

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
os.makedirs('results', exist_ok=True)

def smooth(v, w=10):
    return np.convolve(v, np.ones(w)/w, mode='valid').tolist() if len(v)>=w else v
def cum_mean(v):
    s, o = 0.0, []
    for i,x in enumerate(v): s+=x; o.append(s/(i+1))
    return o

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0f172a')
for row in axes:
    for ax in row:
        ax.set_facecolor('#1e293b')
        ax.tick_params(colors='#94a3b8')
        ax.xaxis.label.set_color('#94a3b8'); ax.yaxis.label.set_color('#94a3b8')
        ax.title.set_color('#e2e8f0')
        for s in ax.spines.values(): s.set_color('#334155')
R, G, Y = '#ef4444', '#22c55e', '#f59e0b'

# Panel 1: Cumulative mean
ax = axes[0][0]
ax.plot(cum_mean(baseline_rewards), color=R, lw=2.5, label='Baseline (random)')
ax.plot(cum_mean(trained_rewards), color=G, lw=2.5, label='Trained (UCB1+Bayes)')
ax.axhline(0, color='#475569', ls='--', lw=0.8)
ax.set_xlabel('Episode'); ax.set_ylabel('Cumulative Mean Reward')
ax.set_title('Learning Curve', fontsize=11, fontweight='bold')
ax.legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='#e2e8f0', fontsize=9)

# Panel 2: Trained zoomed
ax = axes[0][1]
st = smooth(trained_rewards, 5)
ax.plot(st, color=G, lw=2); ax.fill_between(range(len(st)), st, alpha=0.15, color=G)
ax.axhline(0, color='#475569', ls='--', lw=0.8)
for i in range(0, len(trained_rewards), 20):
    c = trained_rewards[i:i+20]; wm = sum(c)/len(c)
    ax.plot(i+10, wm, 'o', color=Y, ms=8)
    ax.annotate(f'{wm:.2f}', (i+10,wm), textcoords='offset points', xytext=(0,12),
               color=Y, fontsize=9, ha='center', fontweight='bold')
ax.set_xlabel('Episode'); ax.set_ylabel('Reward (smoothed)')
ax.set_title('Trained Agent (zoomed)', fontsize=11, fontweight='bold')

# Panel 3: Bar comparison
ax = axes[1][0]
bm = sum(baseline_rewards)/len(baseline_rewards)
tm = sum(trained_rewards)/len(trained_rewards)
bars = ax.bar(['Baseline\n(random)','Trained\n(UCB1+Bayes)'], [bm,tm], color=[R,G], edgecolor='#e2e8f0', lw=0.5, width=0.45)
for bar,val in zip(bars,[bm,tm]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+(0.3 if val>=0 else -1.2),
            f'{val:.2f}', ha='center', va='bottom', color='#e2e8f0', fontsize=14, fontweight='bold')
pct = ((tm-bm)/abs(bm))*100 if bm!=0 else 0
ax.set_ylabel('Mean Reward'); ax.set_title(f'Mean Reward ({pct:+.1f}% improvement)', fontsize=11, fontweight='bold')
ax.axhline(0, color='#475569', ls='--', lw=0.8)

# Panel 4: UCB1 arms
ax = axes[1][1]
arms = bandit.arm_stats()[:8]
names = [a['arm'].split('/')[1] for a in reversed(arms)]
pulls = [a['pulls'] for a in reversed(arms)]
ax.barh(names, pulls, color=G, edgecolor='#e2e8f0', lw=0.3, alpha=0.85)
for i,p in enumerate(pulls): ax.text(p+max(pulls)*0.02, i, str(p), va='center', color='#e2e8f0', fontsize=9)
ax.set_xlabel('Total Pulls'); ax.set_title('UCB1 Arm Selection (Auer 2002)', fontsize=11, fontweight='bold')

fig.suptitle('SENTINEL Training Results (200 episodes)', color='#f8fafc', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
print('Plot saved to results/training_curves.png')
plt.show()

## Cell 6 — Before/After Behavior Transcript

In [ ]:
import json, os
from sentinel.env import Sentinel_Env
from sentinel.training.pipeline import _get_action, TrainingConfig

def capture(env, action_fn, label, n_steps=10):
    lines = [f'\n{"="*60}', f'  {label}', f'{"="*60}']
    obs, info = env.reset(seed=77)
    lines.append(f'Incident: {info.get("incident_id","?")}')
    r = env.render()
    if r: lines.append(r)
    lines.append('')
    term = trunc = False; step = 0; cum = 0.0
    while not (term or trunc) and step < n_steps:
        action = action_fn(obs, step)
        obs, rw, term, trunc, info = env.step(action)
        cum += rw
        svc = action.get('params',{}).get('service','-')
        lines.append(f'  Step {step+1:2d} | {action["agent"]:8s} | {action["name"]:20s} | target={svc:20s} | r={rw:+.3f} | cum={cum:+.3f}')
        step += 1
    lines.append(f'\n  Final reward: {cum:+.4f}')
    return '\n'.join(lines)

RA = [
    {'agent':'holmes','category':'investigative','name':'QueryLogs','params':{'service':'api-gateway','time_range':[0,60]}},
    {'agent':'forge','category':'remediation','name':'RestartService','params':{'service':'api-gateway'}},
]
e1 = Sentinel_Env(config_path='env_spec.yaml', incident_library_path='incident_library.yaml', render_mode='human')
before = capture(e1, lambda o,s: RA[s%len(RA)], 'BEFORE (Random Actions)')

e2 = Sentinel_Env(config_path='env_spec.yaml', incident_library_path='incident_library.yaml', render_mode='human')
tcfg = TrainingConfig(agent='holmes')
def trained_fn(obs, step):
    a = _get_action(None, obs, tcfg)
    a.pop('_ucb1_arm_idx', None)
    return a
after = capture(e2, trained_fn, 'AFTER (UCB1 + Bayesian RCA)')

transcript = before + '\n\n' + after
print(transcript)
os.makedirs('results', exist_ok=True)
with open('results/before_after_transcript.md', 'w') as f:
    f.write('# SENTINEL Behavior Transcript\n\n```\n' + transcript + '\n```\n')
print('\nSaved to results/before_after_transcript.md')

## Cell 7 — Results Summary

In [ ]:
import json, os
bm = sum(baseline_rewards)/len(baseline_rewards)
tm = sum(trained_rewards)/len(trained_rewards)
pct = ((tm-bm)/abs(bm))*100 if bm!=0 else 0
print('\n' + '='*60)
print('  SENTINEL — Final Results')
print('='*60)
print(f'{"Metric":<25} {"Baseline":>12} {"Trained":>12}')
print('-'*50)
print(f'{"Mean Reward":<25} {bm:>12.4f} {tm:>12.4f}')
print(f'{"Min Reward":<25} {min(baseline_rewards):>12.4f} {min(trained_rewards):>12.4f}')
print(f'{"Max Reward":<25} {max(baseline_rewards):>12.4f} {max(trained_rewards):>12.4f}')
print(f'{"Improvement":<25} {pct:>11.1f}%')
print('='*50)
os.makedirs('results', exist_ok=True)
with open('results/simulation_results.json', 'w') as f:
    json.dump({'baseline':{'mean':bm,'rewards':baseline_rewards},'trained':{'mean':tm,'rewards':trained_rewards}}, f, indent=2)
print('\nAll results saved to results/')